# HAP-E — GoEmotions processing — GPU inference path

Sentence-level GoEmotions tagging for the **HAP-E** corpus (human-vs-AI parallel corpus), computed with [`cirimus/modernbert-large-go-emotions`](https://huggingface.co/cirimus/modernbert-large-go-emotions). The corpus is *long*: one row per `(author, doc_id)` with a single `text` column, where `author` is the model label (`human` plus 12 LLM groups) and `doc_id` = `{author}_{genre}_{NNNN}` across 6 genres (`acad, blog, fic, news, spok, tvm`). Raw text is pulled per-model from HuggingFace (`browndw/human-ai-parallel-corpus-2`), mirroring `Pybiber_HAP-E.ipynb` cell 0. Each document is sentence-tokenized, and every sentence keeps its full **28-dim sigmoid probability vector** (27 emotions + `neutral`) — nothing is collapsed to an argmax label.

**This is the processing notebook.** It produces the artifacts below; the analysis and plots live in [4c.GoEmotions_analysis_Mac.ipynb](4c.GoEmotions_analysis_Mac.ipynb).

Counts: for each sentence (NLTK Punkt), if an emotion is ≥ .30 it gets a +1 for that sentence (multi-label, so several emotions in one sentence are captured). The .30 cut is Google's published GoEmotions threshold (`threshold` in `calculate_metrics.py`, `eval_prob_threshold` in `bert_classifier.py`; the GoEmotions team publishes no per-emotion cuts). Per-label tuned thresholds for the model we actually run — used downstream by 6/6a/website — are applied separately by `4b.goemotions_perlabel_threshold_counts.py`.

Artifacts written to `data_processed/`:
- **`goemotions_sentence_probs.parquet`** — one row per sentence (`_row`, `doc_id`, `author`, `genre`, `base_id`, `sent_idx`, `sentence`, 28 `prob_<emotion>` columns). Primary output; consumed by 4b, 6, 6a, and the website notebook.
- **`goemotions_probs.parquet`** — per-doc mean sigmoid probability (`emo_<emotion>` columns + metadata). Default source for 4c.
- **`goemotions_sentence_frac.parquet`** — per-doc fraction of sentences firing each emotion (flat 0.30, Google's cut). Alternate source for 4c.
- **`goemotions_sentence_counts_threshold.parquet`** — per-doc flat-0.30 sentence counts (baseline). The per-label-threshold counts the dotplots use come from `4b`.

`base_id` (e.g. `acad_0001`) strips the author prefix and is the **parallel key** aligning each author on the same source sample — used by the contrast analyses downstream.

**Scope knob:** `N_PER_AUTHOR` subsamples per author (full corpus = 106k docs is heavy GPU work). Default ≈ 200/author across all genres; set `None` for the full corpus.

Target machine: Windows 11 with NVIDIA GPU. Fails loud if CUDA is unavailable. Requires `transformers>=4.48` (ModernBERT), a CUDA `torch`, `huggingface_hub` (for the `hf://` reads), `tqdm`, `pyarrow`. Sentence splitting prefers `nltk` Punkt, regex fallback.

*Notebook scaffolding authored by Claude.*

In [ ]:
import time
_t0 = time.time()
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import pyarrow.parquet as pq
import huggingface_hub  # registers the hf:// fsspec handler used by pd.read_parquet below

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

print(f'[{time.time()-_t0:6.2f}s] imports')
print(f'torch={torch.__version__}  cuda_available={torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available — this notebook targets the Windows GPU machine.')
print(f'device={torch.cuda.get_device_name(0)}')

MODEL_NAME  = 'cirimus/modernbert-large-go-emotions'
DEVICE      = 'cuda'

# --- Sentence-level GoEmotions PROBABILITIES ---------------------------------
# Each document is split into sentences; every sentence keeps its full 28-dim
# sigmoid probability vector (NOT argmax, NOT counts). Artifacts:
#   * sentence table  -> one row per sentence, 28 prob_<emotion> columns (persisted)
#   * emo_matrix      -> per-doc MEAN sigmoid prob across that doc's sentences
#                        (persisted as goemotions_probs.parquet for the Mac notebook).
SENT_CACHE_PATH = 'data_processed/goemotions_sentence_probs.parquet'

MAX_LENGTH  = 1024
BATCH_SIZE  = 32
USE_FP16    = True

RNG_SEED    = 0

## 1. HAP-E authors and source text

The 13 author labels and their per-model text parquets on HuggingFace, copied verbatim from `Pybiber_HAP-E.ipynb` cell 0. Humans use `chunk_2` only. `N_PER_AUTHOR` controls the subset size.

In [ ]:
# === HAP-E authors (13) and the per-model HuggingFace text parquets ==========
# Author label -> filename under HF_TEXT_BASE. Mirrors Pybiber_HAP-E.ipynb cell 0.
MODEL_FILES = {
    "human":                "hape2-text_chunk_2.parquet",
    "gpt-4o":               "hape2-text_gpt-4o-2024-08-06.parquet",
    "gpt-4o-mini":          "hape2-text_gpt-4o-mini-2024-07-18.parquet",
    "gpt-5-mini":           "hape2-text_gpt-5-mini-2025-08-07.parquet",
    "llama-3-70b":          "hape2-text_Meta-Llama-3-70B.parquet",
    "llama-3-70b-instruct": "hape2-text_Meta-Llama-3-70B-Instruct.parquet",
    "llama-3-8b":           "hape2-text_Meta-Llama-3-8B.parquet",
    "llama-3-8b-instruct":  "hape2-text_Meta-Llama-3-8B-Instruct.parquet",
    "gemma-2-9b":           "hape2-text_gemma-2-9b.parquet",
    "gemma-2-9b-it":        "hape2-text_gemma-2-9b-it.parquet",
    "gemma-2-27b":          "hape2-text_gemma-2-27b.parquet",
    "gemma-2-27b-it":       "hape2-text_gemma-2-27b-it.parquet",
    "claude-haiku-4-5":     "hape2-text_claude-haiku-4-5-20251001.parquet",
}
HF_TEXT_BASE = "hf://datasets/browndw/human-ai-parallel-corpus-2/text_data/"

# Subset knob: GoEmotions sentence inference over the full 106k-doc corpus is heavy.
# N_PER_AUTHOR samples this many docs per author (drawn across all genres); None = full corpus.
N_PER_AUTHOR = 200

print(f'Defined {len(MODEL_FILES)} HAP-E authors; N_PER_AUTHOR={N_PER_AUTHOR}')

## 2a. Preflight — environment, GPU, HF reachability, model smoke test

Confirms the Windows GPU machine has the right packages (`transformers>=4.48` for ModernBERT, a CUDA-built `torch`), reports GPU name + VRAM, checks that the HAP-E text dataset is reachable on HuggingFace by reading one model parquet's schema, and runs a 4-sample forward pass through `cirimus/modernbert-large-go-emotions` with fp16 autocast. Prints top-3 predicted labels per sample so model + tokenizer compatibility can be sanity-checked at a glance. Raises with an explicit fix hint on failure.

*Authored by Claude.*

In [ ]:
import sys
import importlib.metadata as ilmd
from packaging.version import Version

def _ver(pkg):
    try:
        return ilmd.version(pkg)
    except ilmd.PackageNotFoundError:
        return None

print(f'Python:         {sys.version.split()[0]}')
print(f'torch:          {_ver("torch")}  (cuda build: {torch.version.cuda})')
print(f'transformers:   {_ver("transformers")}')
print(f'tokenizers:     {_ver("tokenizers")}')
print(f'accelerate:     {_ver("accelerate")}')
print(f'huggingface_hub:{_ver("huggingface_hub")}')
print(f'pyarrow:        {_ver("pyarrow")}')
print(f'tqdm:           {_ver("tqdm")}')

tf_ver = _ver('transformers')
if tf_ver is None or Version(tf_ver) < Version('4.48.0'):
    raise RuntimeError(f'transformers>=4.48 required for ModernBERT (have {tf_ver}). '
                       'Fix: pip install -U "transformers>=4.48"')
if torch.version.cuda is None:
    raise RuntimeError('Installed torch was built without CUDA. Reinstall the CUDA wheel, e.g.: '
                       'pip install --index-url https://download.pytorch.org/whl/cu121 torch')

props = torch.cuda.get_device_properties(torch.cuda.current_device())
print(f'\nGPU:            {props.name}')
print(f'  compute cap:  {props.major}.{props.minor}')
print(f'  total VRAM:   {props.total_memory / 1024**3:.1f} GB')

# HAP-E text lives on HuggingFace (no local text parquet). Confirm one model file is reachable
# and exposes the doc_id/text columns the loader needs.
_probe = HF_TEXT_BASE + MODEL_FILES['human']
try:
    _names = set(pq.read_schema(_probe).names)
except Exception as e:
    raise RuntimeError(f'Could not read {_probe} ({type(e).__name__}: {e}). '
                       'Check network access and that huggingface_hub is installed.')
_need = {'doc_id', 'text'} - _names
if _need:
    raise RuntimeError(f'{_probe} missing required columns: {_need} (have {sorted(_names)})')
print(f'\nHAP-E source: {_probe} OK  (columns include doc_id, text)')

print('\nLoading tokenizer + model for smoke test...')
_t = time.time()
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
_mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f'  loaded in {time.time()-_t:.1f}s')

_samples = ['I love this!', 'I am so angry.', 'That is just confusing.', 'Neutral statement of fact.']
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.float16, enabled=USE_FP16):
    _enc = _tok(_samples, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
    _t = time.time()
    _logits = _mdl(**_enc).logits
    torch.cuda.synchronize()
print(f'  forward pass on {len(_samples)} samples: {(time.time()-_t)*1000:.0f} ms')
print(f'  num_labels:   {_mdl.config.num_labels}')
print(f'  peak VRAM:    {torch.cuda.max_memory_allocated() / 1024**2:.0f} MB (batch of {len(_samples)})')

_probs = torch.sigmoid(_logits.float()).cpu().numpy()
for s, p in zip(_samples, _probs):
    _top = np.argsort(p)[::-1][:3]
    print(f'  "{s}"  -> ' + ', '.join(f'{_mdl.config.id2label[i]}={p[i]:.2f}' for i in _top))

del _mdl, _tok, _enc, _logits
torch.cuda.empty_cache()

print('\nPREFLIGHT OK — safe to run inference.')

## 2. Load HAP-E text (long format)

Read each model's text parquet from HuggingFace, prefix `doc_id` with the author label, concatenate, and derive `genre` (between the first and second underscore) and `base_id` (the author-stripped parallel key). The frame is already long — one row per `(author, doc_id)` — so no melt is needed. `N_PER_AUTHOR` then subsamples per author. `_row` is each row's positional index, used later to slice into `emo_matrix`.

In [ ]:
def _base_id(doc_id, author):
    """Strip the '{author}_' prefix and any '@...' suffix -> the parallel key (e.g. 'acad_0001')."""
    s = re.sub(rf'^{re.escape(author)}_', '', doc_id)
    return re.sub(r'@.+$', '', s)

_t = time.time()
parts = []
for label, fname in MODEL_FILES.items():
    frame = pd.read_parquet(HF_TEXT_BASE + fname, columns=['doc_id', 'text'])
    frame['author'] = label
    # Keep the HF doc_id's embedded metadata: it carries an '@'-suffix (@chunk_2 for human,
    # @<model>-<date> for each LLM). The prefixed doc_id therefore looks like
    # 'gpt-4o_acad_0001@gpt-4o-2024-08-06' -- the source tag is preserved in the text name.
    frame['doc_id'] = label + '_' + frame['doc_id'].astype(str)
    parts.append(frame)
src = pd.concat(parts, ignore_index=True)
print(f'[{time.time()-_t:6.2f}s] read {len(MODEL_FILES)} HF text parquets -> shape={src.shape}')

src['genre']      = src['doc_id'].str.split('_').str[1]                 # between 1st and 2nd '_'
src['base_id']    = [_base_id(d, a) for d, a in zip(src['doc_id'], src['author'])]
src['source_tag'] = src['doc_id'].str.extract(r'@(.+)$')[0].fillna('')  # @-suffix metadata
src = src[src['text'].notna() & (src['text'].astype(str).str.len() > 0)].reset_index(drop=True)

# Subsample N_PER_AUTHOR docs per author (drawn across all genres), deterministic via RNG_SEED.
if N_PER_AUTHOR is not None:
    src = (src.groupby('author', group_keys=False)
              .apply(lambda g: g.sample(min(len(g), N_PER_AUTHOR), random_state=RNG_SEED))
              .reset_index(drop=True))

df = src.reset_index(drop=True)
df['_row'] = np.arange(len(df))
print(f'[{time.time()-_t:6.2f}s] HAP-E long frame -> shape={df.shape}')
print(df['author'].value_counts())
print('genres:', df['genre'].value_counts().to_dict())
print('source_tag examples:', df.groupby('author')['source_tag'].first().to_dict())

## 3. GoEmotions inference (sentence-level probabilities)

Loads `cirimus/modernbert-large-go-emotions`, sentence-tokenizes every document (nltk Punkt, regex fallback), runs batched fp16 inference under `torch.amp.autocast` over the flattened sentence list, and keeps each sentence's **full 28-dim sigmoid probability vector** â€” no argmax, no counts. The per-sentence probabilities are assembled into `sent_table` (one row per sentence) for persistence. For the downstream comparison/plot sections, sentences are averaged within each document into `emo_matrix` (shape `n_docs Ã— 28`), the per-document **mean sentence probability**. Documents that tokenize to zero sentences map to an all-zero row. If `goemotions_sentence_probs.parquet` already exists and its `_row` indices fit the current melt, inference is skipped and `emo_matrix` is rebuilt from the cached per-sentence probs.

In [ ]:
def _build_sentence_splitter():
    """Prefer nltk's Punkt; fall back to a regex splitter if nltk/punkt is unavailable."""
    try:
        import nltk
        have = False
        for pkg in ('punkt_tab', 'punkt'):
            try:
                nltk.data.find(f'tokenizers/{pkg}'); have = True; break
            except LookupError:
                continue
        if not have:
            for pkg in ('punkt_tab', 'punkt'):
                try:
                    nltk.download(pkg, quiet=True); have = True; break
                except Exception:
                    continue
        from nltk.tokenize import sent_tokenize
        sent_tokenize('Probe one. Probe two.')  # force a real call to surface missing data
        print('Sentence splitter: nltk Punkt')
        return sent_tokenize
    except Exception as e:
        print(f'Sentence splitter: regex fallback ({type(e).__name__}: {e})')
        _re = re.compile(r'(?<=[.!?])\s+')
        return lambda t: _re.split(t)

def _split_sentences(text, splitter):
    return [s.strip() for s in splitter(str(text)) if s and s.strip()]

def _load_sentence_cache():
    """Reload the per-sentence prob table and rebuild the per-doc mean matrix aligned to df['_row']."""
    if not Path(SENT_CACHE_PATH).exists():
        return None, None, None
    cached = pd.read_parquet(SENT_CACHE_PATH)
    if '_row' not in cached.columns:
        print('Cache missing _row column -> recomputing'); return None, None, None
    prob_cols = [c for c in cached.columns if c.startswith('prob_')]
    emotions  = [c[5:] for c in prob_cols]
    n = len(df)
    grp    = cached.groupby('_row')
    means  = grp[prob_cols].mean()
    counts = grp.size()
    if len(means) == 0 or int(means.index.max()) >= n:
        print(f'Cache _row range mismatch (max={int(means.index.max()) if len(means) else "n/a"} vs n={n}) -> recomputing')
        return None, None, None
    doc_mean = np.zeros((n, len(emotions)), dtype=np.float32)
    n_sent   = np.zeros(n, dtype=np.int64)
    doc_mean[means.index.values] = means.values.astype(np.float32)
    n_sent[counts.index.values]  = counts.values
    return doc_mean, emotions, n_sent

_t = time.time()
emo_matrix, EMOTIONS, doc_sent_counts = _load_sentence_cache()
sent_table = None
if emo_matrix is not None:
    print(f'[{time.time()-_t:6.2f}s] loaded sentence cache -> per-doc matrix shape={emo_matrix.shape}')
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    EMOTIONS = [model.config.id2label[i] for i in range(model.config.num_labels)]
    print(f'Labels ({len(EMOTIONS)}): {EMOTIONS}')

    splitter = _build_sentence_splitter()

    # Sentence-tokenize every doc, remembering its source row and within-doc index.
    doc_sent_counts = np.zeros(len(df), dtype=np.int64)
    flat_sentences, flat_doc_idx, flat_sent_idx = [], [], []
    for row_idx, text in enumerate(df['text'].astype(str).tolist()):
        sents = _split_sentences(text, splitter)
        doc_sent_counts[row_idx] = len(sents)
        for k, s in enumerate(sents):
            flat_sentences.append(s)
            flat_doc_idx.append(row_idx)
            flat_sent_idx.append(k)
    flat_doc_idx  = np.asarray(flat_doc_idx,  dtype=np.int64)
    flat_sent_idx = np.asarray(flat_sent_idx, dtype=np.int64)
    print(f'{len(df)} docs -> {len(flat_sentences)} sentences '
          f'(mean {len(flat_sentences)/max(1, len(df)):.1f}/doc)')

    # Full 28-dim sigmoid probability vector per sentence — nothing collapsed.
    sent_probs = np.empty((len(flat_sentences), len(EMOTIONS)), dtype=np.float32)
    with torch.inference_mode():
        for i in tqdm(range(0, len(flat_sentences), BATCH_SIZE), desc='GoEmotions (sentence probs)'):
            batch = flat_sentences[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=USE_FP16):
                logits = model(**enc).logits
            sent_probs[i:i+len(batch)] = torch.sigmoid(logits.float()).cpu().numpy()

    # Per-doc analysis vector = MEAN sentence probability (docs w/ 0 sentences -> zeros).
    emo_matrix = np.zeros((len(df), len(EMOTIONS)), dtype=np.float32)
    np.add.at(emo_matrix, flat_doc_idx, sent_probs)
    denom = doc_sent_counts.astype(np.float32).copy(); denom[denom == 0] = 1.0
    emo_matrix = emo_matrix / denom[:, None]

    # Per-sentence table to persist (one row per sentence). HAP-E metadata: author/genre/base_id.
    meta = df.iloc[flat_doc_idx][['doc_id', 'author', 'genre', 'base_id', 'source_tag']].reset_index(drop=True)
    sent_table = meta.copy()
    sent_table.insert(0, '_row', flat_doc_idx)
    sent_table['sent_idx'] = flat_sent_idx
    sent_table['sentence'] = flat_sentences
    for j, e in enumerate(EMOTIONS):
        sent_table[f'prob_{e}'] = sent_probs[:, j]

    print(f'[{time.time()-_t:6.2f}s] inference done -> per-doc matrix {emo_matrix.shape}, '
          f'sentence table {sent_table.shape}')

    del model, tokenizer
    torch.cuda.empty_cache()

EMO_COLS  = [f'emo_{e}'  for e in EMOTIONS]   # used by the per-doc-mean analysis cells
PROB_COLS = [f'prob_{e}' for e in EMOTIONS]

### 3a. Persist per-sentence probability vectors

Writes one row per sentence to `goemotions_sentence_probs.parquet`: `_row` (the source document's positional index), HAP-E metadata (`doc_id`, `author`, `genre`, `base_id`), `sent_idx` (within-document sentence position), the sentence text, and 28 `prob_<emotion>` columns holding the raw sigmoid probabilities. Skips if the file already exists, or if the table was loaded from cache (nothing new to write).

In [ ]:
if Path(SENT_CACHE_PATH).exists():
    print(f'{SENT_CACHE_PATH} already exists -> not overwriting')
elif sent_table is None:
    print('Sentence table not in memory (loaded from cache) -> nothing to write')
else:
    os.makedirs('data_processed', exist_ok=True)
    sent_table.to_parquet(SENT_CACHE_PATH, index=False)
    print(f'Wrote {SENT_CACHE_PATH}: shape={sent_table.shape} '
          f'(one row per sentence, {len(EMOTIONS)} prob_ columns)')

### 3b. Sanity check â€” per-sentence parquet

Reloads `goemotions_sentence_probs.parquet` and asserts: row count equals the total number of sentences, exactly 28 `prob_` columns all within `[0, 1]` (no NaNs), `_row`/`sent_idx` are well-formed, and the per-doc mean of the cached probs reproduces the in-memory `emo_matrix` (the matrix the analysis cells use). Raises on any mismatch so a corrupt/stale cache can't silently feed the plots.

In [ ]:
_chk = pd.read_parquet(SENT_CACHE_PATH)
_prob_cols = [c for c in _chk.columns if c.startswith('prob_')]

# 1. schema: 28 prob_ columns matching EMOTIONS, plus the expected HAP-E metadata columns
assert _prob_cols == PROB_COLS, f'prob columns differ from PROB_COLS: {set(_prob_cols) ^ set(PROB_COLS)}'
_expected_meta = {'_row', 'doc_id', 'author', 'genre', 'base_id', 'source_tag', 'sent_idx', 'sentence'}
assert _expected_meta <= set(_chk.columns), f'missing metadata columns: {_expected_meta - set(_chk.columns)}'

# 2. probabilities are valid: finite and within [0, 1]
_pv = _chk[_prob_cols].to_numpy()
assert np.isfinite(_pv).all(), 'found NaN/inf in prob_ columns'
assert _pv.min() >= 0.0 and _pv.max() <= 1.0, f'prob_ out of [0,1]: min={_pv.min()}, max={_pv.max()}'

# 3. row count == total sentences, and _row / sent_idx are well-formed
assert int(_chk['_row'].min()) >= 0 and int(_chk['_row'].max()) < len(df), '_row out of range vs df'
assert int(_chk['sent_idx'].min()) >= 0, 'negative sent_idx'
_per_doc = _chk.groupby('_row').size()
if doc_sent_counts is not None:
    assert int(_per_doc.sum()) == int(doc_sent_counts.sum()), \
        f'row count {len(_chk)} != total sentences {int(doc_sent_counts.sum())}'
    # every doc with >=1 sentence is represented, with the right count
    _nz = np.nonzero(doc_sent_counts)[0]
    assert _per_doc.reindex(_nz, fill_value=0).to_numpy().tolist() == doc_sent_counts[_nz].tolist(), \
        'per-doc sentence counts in cache disagree with doc_sent_counts'

# 4. per-doc mean of cached probs reproduces emo_matrix (the analysis matrix)
_recon = np.zeros_like(emo_matrix)
_means = _chk.groupby('_row')[_prob_cols].mean()
_recon[_means.index.to_numpy()] = _means.to_numpy(dtype=np.float32)
_max_dev = float(np.abs(_recon - emo_matrix).max())
assert _max_dev < 1e-4, f'reconstructed per-doc mean deviates from emo_matrix by {_max_dev}'

print(f'OK  rows={len(_chk):,}  docs={_chk["_row"].nunique():,}  '
      f'prob range=[{_pv.min():.4f}, {_pv.max():.4f}]  '
      f'mean sents/doc={_per_doc.mean():.1f}  emo_matrix max dev={_max_dev:.2e}')
del _chk, _pv, _recon, _means

In [ ]:
# === 3c. Persist per-doc aggregates for the Mac analysis notebook (4c) ========
# Two per-document artifacts, one row per (doc_id, author) with emo_<emotion> columns:
#   * goemotions_probs.parquet         -> MEAN sentence sigmoid probability (4c default)
#   * goemotions_sentence_frac.parquet -> FRACTION of sentences firing each emotion
#                                         (flat 0.30 = Google's published GoEmotions cut)
# Metadata carried: doc_id, author, genre, base_id, n_sentences.
os.makedirs('data_processed', exist_ok=True)
_meta_doc = df[['doc_id', 'author', 'genre', 'base_id', 'source_tag']].copy()
_meta_doc['n_sentences'] = doc_sent_counts

# probs: per-doc mean sentence probability (rebuilds emo_matrix into emo_<e> columns).
probs_doc = _meta_doc.copy()
for j, e in enumerate(EMOTIONS):
    probs_doc[f'emo_{e}'] = emo_matrix[:, j]
probs_doc.to_parquet('data_processed/goemotions_probs.parquet', index=False)

# frac: per-doc fraction of sentences firing each emotion at Google's flat 0.30 cut
# (google-research/goemotions: `threshold` in calculate_metrics.py, `eval_prob_threshold`
# in bert_classifier.py -- one global cut; no per-emotion thresholds are published there).
GOOGLE_THRESHOLD = 0.30
_sp = pd.read_parquet(SENT_CACHE_PATH)
_fires = (_sp[PROB_COLS].to_numpy() >= GOOGLE_THRESHOLD).astype(np.float64)
_fd = pd.DataFrame(_fires, columns=[f'emo_{e}' for e in EMOTIONS])
_fd['_row'] = _sp['_row'].values
_frac_by_row = _fd.groupby('_row')[[f'emo_{e}' for e in EMOTIONS]].mean()
_fr = np.zeros((len(df), len(EMOTIONS)), dtype=np.float64)
_fr[_frac_by_row.index.to_numpy()] = _frac_by_row.to_numpy()
frac_doc = _meta_doc.copy()
for j, e in enumerate(EMOTIONS):
    frac_doc[f'emo_{e}'] = _fr[:, j]
frac_doc.to_parquet('data_processed/goemotions_sentence_frac.parquet', index=False)

print(f'wrote goemotions_probs.parquet {probs_doc.shape} and '
      f'goemotions_sentence_frac.parquet {frac_doc.shape} (flat threshold {GOOGLE_THRESHOLD})')
del _sp, _fires, _fd, _frac_by_row, _fr

## 4. Threshold-based sentence counts (flat 0.30, Google baseline)

Regenerate per-document multi-label sentence counts from the cached per-sentence probabilities: a sentence "fires" emotion *e* iff its sigmoid `prob_e ≥ 0.30` (multi-label — a sentence can fire zero, one, or several emotions). Grouped by `(doc_id, author, genre, base_id)` and written to `goemotions_sentence_counts_threshold.parquet`.

This is the **flat-0.30 baseline**, using the cutoff [google-research/goemotions](https://github.com/google-research/google-research/tree/master/goemotions) publishes (`threshold = 0.3` in `calculate_metrics.py`, `eval_prob_threshold = 0.3` in `bert_classifier.py`). Google applies one global cut to every label — they publish no per-emotion thresholds. The **per-label tuned thresholds** that the analysis dotplots and the qualitative notebooks (6, 6a, website) use come from the model card of the checkpoint we actually run (`cirimus/modernbert-large-go-emotions`, "Optimal Results") and are produced separately by `4b.goemotions_perlabel_threshold_counts.py` → `goemotions_sentence_counts_threshold_perlabel.parquet`.

> **Analysis & plots live in [4c.GoEmotions_analysis_Mac.ipynb](4c.GoEmotions_analysis_Mac.ipynb).** This notebook stops at the processing artifacts above; run 4c (Mac/CPU) for the overview heatmap and the Figure-3 relative-usage dotplots (human vs machine).

*Authored by Claude.*

In [ ]:
# === Threshold-based multi-label sentence counts (flat 0.30, from cached probs) =====
# A sentence fires emotion e iff its sigmoid prob_e >= EMO_THRESHOLD (flat 0.30, Google's
# published GoEmotions cut). Multi-label: a sentence can fire 0, 1, or several emotions.
# Built from goemotions_sentence_probs.parquet, grouped by (doc_id, author, genre, base_id),
# written to the flat-0.30 baseline parquet.
# (The dotplots in 4c read the per-label file from 4b.goemotions_perlabel_threshold_counts.py.)
_SENT_PROBS_PATH    = 'data_processed/goemotions_sentence_probs.parquet'
_THRESH_COUNTS_PATH = 'data_processed/goemotions_sentence_counts_threshold.parquet'

# google-research/goemotions: `threshold` (calculate_metrics.py) and `eval_prob_threshold`
# (bert_classifier.py) both default to 0.3, applied globally -- no per-emotion cuts are published.
EMO_THRESHOLD = 0.30   # flat probability threshold for a sentence to "fire" an emotion

_sp = pd.read_parquet(_SENT_PROBS_PATH)
_fires = (_sp[[f'prob_{e}' for e in EMOTIONS]].to_numpy() >= EMO_THRESHOLD).astype(np.int64)
_fd = pd.DataFrame(_fires, columns=[f'emo_{e}' for e in EMOTIONS])
_keys = ['doc_id', 'author', 'genre', 'base_id', 'source_tag']
_fd[_keys] = _sp[_keys].values
thresh_counts = (_fd.groupby(_keys, as_index=False)
                    [[f'emo_{e}' for e in EMOTIONS]].sum())
thresh_counts.to_parquet(_THRESH_COUNTS_PATH, index=False)

_M = thresh_counts[[f'emo_{e}' for e in EMOTIONS]].to_numpy()
print(f'Wrote {_THRESH_COUNTS_PATH}: {thresh_counts.shape} (flat threshold {EMO_THRESHOLD})')
print(f'mean emotions fired/doc {_M.sum(1).mean():.2f}  |  '
      f'sentences firing nothing {(_fires.sum(1) == 0).mean():.1%}  |  '
      f'firing >=2 {(_fires.sum(1) >= 2).mean():.1%}')
print(f'per-emotion mean count/doc range {_M.mean(0).min():.4f}-{_M.mean(0).max():.3f}')
del _sp, _fires, _fd